In [2]:
!git clone https://github.com/MuhammadAhmadIshtiaq/ml-internship-muhammadahmadishtiaq.git
%cd ml-internship-muhammadahmadishtiaq

Cloning into 'ml-internship-muhammadahmadishtiaq'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 144 (delta 53), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (144/144), 1.86 MiB | 11.08 MiB/s, done.
Resolving deltas: 100% (53/53), done.
/content/ml-internship-muhammadahmadishtiaq/ml-internship-muhammadahmadishtiaq


# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAhmadIshtiaq/ml-internship-muhammadahmadishtiaq/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row = one content item (one page) — same grain as ML-03, `content_id` unique per row.

**Time window:** every numeric column is a *trailing 90-day snapshot ending at export time* — this file has no per-row date column, it's one flat export, not a time series. Inside that 90-day window there are two named sub-windows used for the trend/label calculation: `impressions_last_30d` etc. (the most recent 30 days) and `impressions_prev_30d` etc. (the 30 days before that, days 31-60 back). So: one snapshot window (90d) containing two non-overlapping 30-day comparison windows inside it, per the data dictionary.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Verify: one row = one content item
print("Rows:", len(df), "| Unique content_id:", df["content_id"].nunique())
print("Grain holds:", len(df) == df["content_id"].nunique())

# Verify: this file has no real date column (single flat snapshot, not a panel)
date_like_cols = [c for c in df.columns if "date" in c.lower()]
print("Date-like columns found:", date_like_cols, "(none are calendar dates -- confirms flat snapshot)")


Rows: 30000 | Unique content_id: 30000
Grain holds: True
Date-like columns found: ['days_since_last_update'] (none are calendar dates -- confirms flat snapshot)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Context (grouping/joining only, never model inputs):** `content_id`, `client_id` — pseudonymized IDs, 32 distinct clients.

**Label / proxy (the target, or what it's computed from — never a feature):** `is_declining_label` (the target, built downstream), `trend_direction`, `trend_pct` — these three are the label's own computation chain.

**Excluded (not label, not usable as a feature — each with a why):**
- `impressions_last_30d`, `impressions_prev_30d` — these are the *exact numeric inputs* to `trend_pct` (`(last30 − prev30) / prev30 × 100`). Keeping them as features would let a model near-perfectly reconstruct the label instead of learning a real pattern — one arithmetic step of leakage away from the target.
- `provider_used`, `model_used` — the data dictionary flags both as "not a model feature": they describe which LLM generated the article, an internal production detail, not a performance signal a content reviewer would use, and using it risks the model learning provider-specific quirks instead of real content quality.

**Feature (knowable before prediction time, safe to use):** `word_count`, `char_count`, `content_age_days`, `days_since_last_update`, `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions`, `clicks_last_30d`, `clicks_prev_30d`, `sessions_last_30d`, `sessions_prev_30d`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `search_volume`, `competition`, `competition_level`, `cpc`, `content_type`, `main_intent`, `age_tier`, `freshness_tier`, `word_count_tier`, `char_count_tier`, `impression_tier`, `position_tier`. (`clicks_last_30d`/`clicks_prev_30d` and `sessions_last_30d`/`sessions_prev_30d` are kept — they're correlated with the label window but are a *different metric* than the one the label formula uses, so they're a real signal, not a direct leak; still worth re-checking for leakage in the ML-05 audit.)

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

context_cols = ["content_id", "client_id"]
label_chain_cols = ["is_declining_label", "trend_direction", "trend_pct"]
excluded_cols = ["impressions_last_30d", "impressions_prev_30d", "provider_used", "model_used"]

all_cols = set(df.columns) | {"is_declining_label"}
accounted_for = set(context_cols) | set(label_chain_cols) | set(excluded_cols)
feature_cols = sorted(all_cols - accounted_for)

print("Context:", len(context_cols))
print("Label chain:", len(label_chain_cols))
print("Excluded:", len(excluded_cols))
print("Feature candidates:", len(feature_cols))
print(feature_cols)


Context: 2
Label chain: 3
Excluded: 4
Feature candidates: 36
['age_tier', 'age_tier_order', 'ai_sessions_90d', 'ai_traffic_pct', 'avg_position', 'char_count', 'char_count_tier', 'clicks_90d', 'clicks_last_30d', 'clicks_prev_30d', 'competition', 'competition_level', 'content_age_days', 'content_type', 'cpc', 'ctr', 'days_since_last_update', 'days_with_impressions', 'days_with_sessions', 'engaged_sessions_90d', 'engagement_rate', 'freshness_tier', 'impression_tier', 'impressions_90d', 'main_intent', 'pageviews_90d', 'position_tier', 'scroll_events_90d', 'scroll_rate', 'search_volume', 'sessions_90d', 'sessions_last_30d', 'sessions_prev_30d', 'users_90d', 'word_count', 'word_count_tier']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Four checks below, one per contract claim: grain, the leakage claim on the 30-day windows, missingness pattern by `content_type`, and the two named sub-windows actually being non-overlapping halves of the 90-day total.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# --- Check 1: grain (already shown in section 1, restated here as a pass/fail) ---
grain_ok = df["content_id"].value_counts().max() == 1
print("Grain check (no content_id repeats):", grain_ok)

# --- Check 2: prove the leakage claim -- recompute trend_pct from the two 'last30/prev30' cols ---
recomputed = (df["impressions_last_30d"] - df["impressions_prev_30d"]) / df["impressions_prev_30d"].replace(0, float("nan")) * 100
match_rate = (recomputed.round(1) == df["trend_pct"]).mean()
print(f"Share of rows where recomputed trend_pct matches exactly: {match_rate:.1%}")
print("(High match confirms these two columns ARE the label formula's inputs -> correctly excluded.)")

# --- Check 3: missingness follows content_type, not random ---
missing_by_type = df.groupby("content_type")["search_volume"].apply(lambda s: s.isna().mean())
print("\nsearch_volume missingness by content_type:")
print(missing_by_type.round(2))

# --- Check 4: last_30 + prev_30 should not double-count against the 90d total ---
print("\nMedian impressions_90d:", df["impressions_90d"].median())
print("Median (last_30 + prev_30):", (df["impressions_last_30d"] + df["impressions_prev_30d"]).median())
print("These are sub-windows of the 90d total, not the full 90d themselves (90d includes an older 30d slice too).")


Grain check (no content_id repeats): True
Share of rows where recomputed trend_pct matches exactly: 88.4%
(High match confirms these two columns ARE the label formula's inputs -> correctly excluded.)

search_volume missingness by content_type:
content_type
comparison article    0.00
feedly article        1.00
keyword article       0.01
Name: search_volume, dtype: float64

Median impressions_90d: 731.0
Median (last_30 + prev_30): 371.0
These are sub-windows of the 90d total, not the full 90d themselves (90d includes an older 30d slice too).


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **No real time series here.** This CSV is one flat 90-day snapshot per page, not a dated panel — it cannot show trajectory over time, only two summary sub-windows. Real per-day history only exists in the gated warehouse release (`fact_content_daily_performance`, per `skills/flyrank/flyrank-data/SKILL.md`), not in this file.
- **Unbalanced client coverage.** This slice covers 32 of 104 total clients in the warehouse, with no way from this file alone to check how long each client's history runs — the warehouse's `dim_clients.gsc_data_start` would be needed for that.
- **`avg_position = 0` means "no data," not rank zero** (1,205 rows) — a naive read would misinterpret these as top-ranked pages.
- **`scroll_rate` and `ai_traffic_pct` can exceed 100%** — their numerator and denominator come from different measurement systems, so they aren't true bounded percentages; treat >100 values as valid, not as data errors.
- **IDs are pseudonyms for grouping only** — `content_id`/`client_id` carry no real-world identity and should never be treated as features or joined against outside data.
- **This is decision-support, not causal proof.** The data can say a page's signals *look like* other declining pages; it can't say what caused any single page's traffic to move.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Quantify two of the limits above directly from the data
no_position_data = (df["avg_position"] == 0).sum()
print("Rows where avg_position == 0 (no data, not rank zero):", no_position_data)

over_100_scroll = (df["scroll_rate"] > 100).sum()
over_100_ai = (df["ai_traffic_pct"] > 100).sum()
print("Rows where scroll_rate > 100%:", over_100_scroll)
print("Rows where ai_traffic_pct > 100%:", over_100_ai)

print("\nClients present in this slice:", df["client_id"].nunique(), "(warehouse has 104 total)")


Rows where avg_position == 0 (no data, not rank zero): 1205
Rows where scroll_rate > 100%: 119
Rows where ai_traffic_pct > 100%: 23

Clients present in this slice: 32 (warehouse has 104 total)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.